In [ ]:
import os
!export LD_LIBRARY_PATH=/usr/lib/x86_64-linux-gnu:$LD_LIBRARY_PATH
os.environ["LD_LIBRARY_PATH"] = "/usr/lib/x86_64-linux-gnu:" + os.environ.get("LD_LIBRARY_PATH", "")
print("CONDA_PREFIX =", os.environ.get("CONDA_PREFIX"))
print("LD_LIBRARY_PATH =", os.environ.get("LD_LIBRARY_PATH"))

In [ ]:
import sys
import os
import importlib.util
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pyvista as pv

print(sys.executable)

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SRC_DIR = REPO_ROOT / "src"
SCRIPTS_DIR = REPO_ROOT / "scripts"

sys.path.insert(0, str(SRC_DIR.resolve()))
sys.path.insert(0, str(SCRIPTS_DIR.resolve()))

from autolabeler.data.csv_loader import H, W, load_csv
from autolabeler.data.schemas import OrganizedLiDARFrame

VIS_SCRIPT = SCRIPTS_DIR / "visualize_point_segmentation.py"
print("Checking for file at:", VIS_SCRIPT.resolve())
print("Does it exist?", VIS_SCRIPT.exists())
if not VIS_SCRIPT.exists():
    raise FileNotFoundError(
        f"Missing {VIS_SCRIPT}. Make sure you are on the branch that contains "
        "scripts/visualize_point_segmentation.py or copy that script into this repo."
    )

spec = importlib.util.spec_from_file_location("visualize_point_segmentation", VIS_SCRIPT)
visualize_point_segmentation = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(visualize_point_segmentation)
load_semantic_labels = visualize_point_segmentation.load_semantic_labels
load_confidence = visualize_point_segmentation.load_confidence
remap_legacy_raw_csv_mask_order = visualize_point_segmentation.remap_legacy_raw_csv_mask_order

print(H, W)
print("REPO_ROOT =", REPO_ROOT.resolve())

In [ ]:
frame_id = "lidar_side_front_udp-D3-2026_01_12_13_16_21_46305"

data_dir = REPO_ROOT / "data"
litept_dir = REPO_ROOT / "output" / "litept" / frame_id

csv_path = data_dir / f"{frame_id}.csv"
semantic_mask_path = litept_dir / "semantic_mask.npy"
confidence_mask_path = litept_dir / "confidence.npy"

print("csv_path =", csv_path)
print("semantic_mask_path =", semantic_mask_path)
print("confidence_mask_path =", confidence_mask_path)

In [ ]:
# Raw fallback, if needed:
# semantic_mask = np.fromfile(semantic_mask_path, dtype=np.uint16)
# confidence_mask = np.fromfile(confidence_mask_path, dtype=np.float32)

# Set True only for masks exported before the raw CSV loader transpose fix.
# If you rerun LitePT after updating csv_loader.py, keep this False.
MASK_WAS_EXPORTED_WITH_OLD_RAW_CSV_LOADER = False

Frame = load_csv(path=str(csv_path), frame_id=frame_id)
points_int = Frame.points_range.reshape(-1, 4).astype(np.float32)
coordinates = points_int[:, :3]

semantic_mask = load_semantic_labels(
    mask_path=str(semantic_mask_path),
    labels_jsonl=None,
    frame_id=frame_id,
    point_count=len(points_int),
    height=H,
    width=W,
    allow_resize=True,
)
confidence_mask = load_confidence(
    str(confidence_mask_path),
    point_count=len(points_int),
    height=H,
    width=W,
    allow_resize=True,
)

if MASK_WAS_EXPORTED_WITH_OLD_RAW_CSV_LOADER:
    semantic_mask = remap_legacy_raw_csv_mask_order(semantic_mask, height=H, width=W)
    confidence_mask = remap_legacy_raw_csv_mask_order(confidence_mask, height=H, width=W)

print(np.shape(np.reshape(semantic_mask, -1))[0] == (H * W))
print(np.shape(np.reshape(confidence_mask, -1))[0] == (H * W))
print("semantic shape =", semantic_mask.shape, "dtype =", semantic_mask.dtype)
print("confidence shape =", confidence_mask.shape, "dtype =", confidence_mask.dtype)

semantic_image = semantic_mask.reshape(H, W)
confidence_image = confidence_mask.reshape(H, W)

In [ ]:
plt.figure(figsize=(14, 4))
plt.imshow(semantic_image, aspect="auto")
plt.title("semantic_mask")
plt.colorbar()
plt.show()

In [ ]:
plt.figure(figsize=(14, 4))
plt.imshow(confidence_image, aspect="auto")
plt.title("confidence_mask")
plt.colorbar()
plt.show()

In [ ]:
print(np.unique(semantic_mask))

In [ ]:
pv_points = pv.PolyData(coordinates)
pv_points["intensity"] = points_int[:, 3]
pv_points["semantic"] = semantic_mask.astype(np.float32)
pv_points["confidence"] = confidence_mask.astype(np.float32)

plotter = pv.Plotter()

default_cam = [
    (-8, 0, 0),
    (35.062525033950806, 0, -1.2258319854736328),
    (0, 0, 1),
]

plotter.add_mesh(
    pv_points,
    render_points_as_spheres=True,
    cmap="plasma",
    show_scalar_bar=True,
    scalars="intensity",
    point_size=5,
)
plotter.camera_position = default_cam

cpos = plotter.show(return_cpos=True)
print(cpos)

In [ ]:
plotter = pv.Plotter()
plotter.add_mesh(
    pv_points,
    render_points_as_spheres=True,
    cmap="tab20",
    show_scalar_bar=True,
    scalars="semantic",
    point_size=5,
)
plotter.camera_position = default_cam

cpos = plotter.show(return_cpos=True)
print(cpos)

In [ ]:
plotter = pv.Plotter()
plotter.add_mesh(
    pv_points,
    render_points_as_spheres=True,
    cmap="viridis",
    show_scalar_bar=True,
    scalars="confidence",
    point_size=5,
)
plotter.camera_position = default_cam

cpos = plotter.show(return_cpos=True)
print(cpos)